<a href="https://colab.research.google.com/github/Rinosa123/Bilingual-Enterprise-RAG-Copilot/blob/main/notebooks/05_end_to_end_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic–English Enterprise RAG Copilot: End-to-End Pipeline

This notebook connects the complete production-style RAG workflow:

1. Multilingual E5 candidate retrieval
2. BGE multilingual cross-encoder reranking
3. Qwen grounded bilingual generation
4. Evidence-only citation validation
5. Safe English and Arabic refusal handling
6. Component and end-to-end latency measurement

In [1]:
%cd /content

# Download the repository or update the existing Colab copy.
!if [ -d "Bilingual-Enterprise-RAG-Copilot/.git" ]; then \
    git -C Bilingual-Enterprise-RAG-Copilot pull --ff-only origin main; \
else \
    git clone https://github.com/Rinosa123/Bilingual-Enterprise-RAG-Copilot.git; \
fi

%cd /content/Bilingual-Enterprise-RAG-Copilot

# Remove Colab's unused Gradio packages to prevent dependency conflicts.
!pip -q uninstall -y gradio gradio-client

# Install the tested model stack.
!pip -q install \
    "transformers==4.57.6" \
    "sentence-transformers==5.6.0" \
    "accelerate>=1.0.0" \
    "bitsandbytes==0.50.0"

/content
Cloning into 'Bilingual-Enterprise-RAG-Copilot'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 98 (delta 40), reused 56 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 84.74 KiB | 5.65 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/Bilingual-Enterprise-RAG-Copilot
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.7 MB/s eta 0:00:00


## 1. Environment and Bilingual Document Loading

Verify the Colab GPU environment and load the synthetic English and Arabic enterprise policy documents using the production ingestion code.

In [2]:
import sys
from pathlib import Path

import bitsandbytes
import sentence_transformers
import torch
import transformers

from src.ingestion.chunker import chunk_documents
from src.ingestion.text_loader import load_text_documents
from src.pipeline import EnterpriseRAGPipeline


PROJECT_ROOT = Path.cwd()
DOCUMENT_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "sample_docs"
)

# Confirm that Colab is using a GPU.
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is unavailable. Select Runtime → "
        "Change runtime type → T4 GPU."
    )

# Load and chunk the bilingual documents.
documents = load_text_documents(
    DOCUMENT_DIRECTORY
)
chunks = chunk_documents(documents)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print(
    "Sentence Transformers:",
    sentence_transformers.__version__,
)
print("Bitsandbytes:", bitsandbytes.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Documents:", len(documents))
print("Chunks:", len(chunks))

print("\nChunks by language:")

for language in ("en", "ar"):
    language_chunks = [
        chunk
        for chunk in chunks
        if chunk.language == language
    ]

    print(
        f"{language}: {len(language_chunks)}"
    )

Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Sentence Transformers: 5.6.0
Bitsandbytes: 0.50.0
CUDA available: True
GPU: Tesla T4
Documents: 2
Chunks: 10

Chunks by language:
en: 5
ar: 5


## 2. Multilingual Dense Retrieval

Load `intfloat/multilingual-e5-small` and create normalized embeddings for every English and Arabic document chunk.

In [3]:
from sentence_transformers import SentenceTransformer


DENSE_MODEL_NAME = (
    "intfloat/multilingual-e5-small"
)

print("Loading:", DENSE_MODEL_NAME)

dense_model = SentenceTransformer(
    DENSE_MODEL_NAME,
    device="cuda",
)

# E5 models expect the "passage:" prefix for documents.
passage_texts = [
    (
        f"passage: {chunk.section}\n"
        f"{chunk.text}"
    )
    for chunk in chunks
]

passage_embeddings = dense_model.encode(
    passage_texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_tensor=True,
    show_progress_bar=True,
)

print("\nDense model loaded successfully.")
print("Model device:", dense_model.device)
print(
    "Embedding shape:",
    tuple(passage_embeddings.shape),
)
print(
    "Embedding dimension:",
    passage_embeddings.shape[1],
)

Loading: intfloat/multilingual-e5-small


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Dense model loaded successfully.
Model device: cuda:0
Embedding shape: (10, 384)
Embedding dimension: 384


### Dense Retriever Adapter

This adapter converts multilingual E5 similarity scores into document chunks accepted by the production `EnterpriseRAGPipeline`. It also records retrieval latency and ranked scores for monitoring.

In [4]:
from collections.abc import Sequence
from time import perf_counter


class DenseRetrieverAdapter:
    """Use multilingual E5 as a pipeline retrieval component."""

    def __init__(
        self,
        model,
        document_chunks: Sequence,
        document_embeddings: torch.Tensor,
    ) -> None:
        self.model = model
        self.document_chunks = tuple(
            document_chunks
        )
        self.document_embeddings = (
            document_embeddings
        )

        self.last_latency_ms = 0.0
        self.last_results = ()

    def __call__(
        self,
        question: str,
        top_k: int,
    ) -> Sequence:
        if top_k < 1:
            raise ValueError(
                "top_k must be at least 1."
            )

        if not self.document_chunks:
            self.last_latency_ms = 0.0
            self.last_results = ()
            return ()

        # Synchronize before timing GPU work.
        torch.cuda.synchronize()
        start_time = perf_counter()

        query_embedding = self.model.encode(
            [f"query: {question}"],
            normalize_embeddings=True,
            convert_to_tensor=True,
            show_progress_bar=False,
        )

        similarity_scores = torch.matmul(
            query_embedding,
            self.document_embeddings.T,
        )[0]

        result_count = min(
            top_k,
            len(self.document_chunks),
        )

        top_scores, top_indices = torch.topk(
            similarity_scores,
            k=result_count,
        )

        index_values = top_indices.tolist()
        score_values = top_scores.tolist()

        ranked_results = tuple(
            (
                self.document_chunks[index],
                float(score),
            )
            for index, score in zip(
                index_values,
                score_values,
            )
        )

        torch.cuda.synchronize()

        self.last_latency_ms = (
            perf_counter() - start_time
        ) * 1000

        self.last_results = ranked_results

        return tuple(
            chunk
            for chunk, _ in ranked_results
        )


dense_retriever = DenseRetrieverAdapter(
    model=dense_model,
    document_chunks=chunks,
    document_embeddings=passage_embeddings,
)


def display_dense_results(
    question: str,
) -> None:
    """Display the top three dense retrieval results."""
    dense_retriever(
        question,
        top_k=3,
    )

    print("=" * 80)
    print("Question:", question)
    print(
        "Latency:",
        f"{dense_retriever.last_latency_ms:.2f} ms",
    )

    for rank, (
        chunk,
        score,
    ) in enumerate(
        dense_retriever.last_results,
        start=1,
    ):
        print(
            f"{rank}. {chunk.chunk_id} | "
            f"score={score:.4f} | "
            f"section={chunk.section}"
        )


display_dense_results(
    "How many annual leave days do "
    "full-time employees receive?"
)

display_dense_results(
    "ما الحد الأقصى لتكلفة الفندق؟"
)

Question: How many annual leave days do full-time employees receive?
Latency: 154.83 ms
1. HR-EN-001-CH-003 | score=0.8941 | section=2. Annual Leave
2. HR-EN-001-CH-002 | score=0.7900 | section=1. Working Hours
3. HR-EN-001-CH-004 | score=0.7808 | section=3. Remote Work
Question: ما الحد الأقصى لتكلفة الفندق؟
Latency: 14.56 ms
1. HR-AR-001-CH-003 | score=0.8903 | section=2. السفر في مهام العمل
2. HR-AR-001-CH-002 | score=0.7946 | section=1. مطالبات المصروفات
3. HR-AR-001-CH-005 | score=0.7727 | section=4. التدريب والتطوير المهني


## 3. Multilingual Cross-Encoder Reranking

Load `BAAI/bge-reranker-v2-m3` directly through Transformers. The reranker jointly examines each question and candidate chunk, producing more precise relevance rankings than embedding similarity alone.

In [5]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)


RERANKER_MODEL_NAME = (
    "BAAI/bge-reranker-v2-m3"
)

print("Loading:", RERANKER_MODEL_NAME)

reranker_tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_MODEL_NAME
)

reranker_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        RERANKER_MODEL_NAME,
        torch_dtype=torch.float16,
    )
    .to("cuda")
    .eval()
)

reranker_memory_gb = (
    reranker_model.get_memory_footprint()
    / (1024 ** 3)
)

print("\nReranker loaded successfully.")
print("Model device:", reranker_model.device)
print(
    "Model memory footprint:",
    f"{reranker_memory_gb:.2f} GB",
)

Loading: BAAI/bge-reranker-v2-m3


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]


Reranker loaded successfully.
Model device: cuda:0
Model memory footprint: 1.06 GB


In [6]:
class BGERerankerAdapter:
    def __init__(
        self,
        tokenizer,
        model,
        max_length: int = 512,
    ) -> None:
        self.tokenizer = tokenizer
        self.model = model
        self.max_length = max_length
        self.last_latency_ms = 0.0
        self.last_results = ()

    def __call__(
        self,
        question: str,
        candidates: Sequence,
        top_k: int,
    ) -> Sequence:
        if top_k < 1:
            raise ValueError("top_k must be at least 1.")

        if not candidates:
            self.last_latency_ms = 0.0
            self.last_results = ()
            return ()

        torch.cuda.synchronize()
        start_time = perf_counter()

        question_passage_pairs = [
            (
                question,
                f"{chunk.section}\n{chunk.text}",
            )
            for chunk in candidates
        ]

        encoded_pairs = self.tokenizer(
            question_passage_pairs,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        encoded_pairs = {
            name: tensor.to(self.model.device)
            for name, tensor in encoded_pairs.items()
        }

        with torch.inference_mode():
            logits = self.model(
                **encoded_pairs
            ).logits.view(-1)

            scores = torch.sigmoid(logits)

        ranked_indices = torch.argsort(
            scores,
            descending=True,
        )

        result_count = min(
            top_k,
            len(candidates),
        )

        ranked_indices = ranked_indices[
            :result_count
        ].tolist()

        score_values = scores.tolist()

        ranked_results = tuple(
            (
                candidates[index],
                float(score_values[index]),
            )
            for index in ranked_indices
        )

        torch.cuda.synchronize()

        self.last_latency_ms = (
            perf_counter() - start_time
        ) * 1000

        self.last_results = ranked_results

        return tuple(
            chunk
            for chunk, _ in ranked_results
        )


bge_reranker = BGERerankerAdapter(
    tokenizer=reranker_tokenizer,
    model=reranker_model,
)

print("BGE reranker adapter created.")

BGE reranker adapter created.


In [7]:
test_question = (
    "كم عدد أيام الإجازة السنوية للموظف؟"
)

dense_candidates = dense_retriever(
    test_question,
    top_k=5,
)

print("Dense candidates:")

for rank, (chunk, score) in enumerate(
    dense_retriever.last_results,
    start=1,
):
    print(
        f"{rank}. {chunk.chunk_id} | "
        f"score={score:.4f} | "
        f"section={chunk.section}"
    )

reranked_chunks = bge_reranker(
    test_question,
    dense_candidates,
    top_k=3,
)

print(
    "\nReranking latency:",
    f"{bge_reranker.last_latency_ms:.2f} ms",
)

print("\nReranked candidates:")

for rank, (chunk, score) in enumerate(
    bge_reranker.last_results,
    start=1,
):
    print(
        f"{rank}. {chunk.chunk_id} | "
        f"score={score:.4f} | "
        f"section={chunk.section}"
    )

Dense candidates:
1. HR-AR-001-CH-004 | score=0.8559 | section=3. الإجازة المرضية
2. HR-AR-001-CH-003 | score=0.8272 | section=2. السفر في مهام العمل
3. HR-AR-001-CH-005 | score=0.8232 | section=4. التدريب والتطوير المهني
4. HR-EN-001-CH-003 | score=0.8102 | section=2. Annual Leave
5. HR-AR-001-CH-002 | score=0.8063 | section=1. مطالبات المصروفات

Reranking latency: 207.51 ms

Reranked candidates:
1. HR-EN-001-CH-003 | score=0.9795 | section=2. Annual Leave
2. HR-AR-001-CH-004 | score=0.0232 | section=3. الإجازة المرضية
3. HR-AR-001-CH-005 | score=0.0070 | section=4. التدريب والتطوير المهني


## 4. Quantized Bilingual Grounded Generation

Load Qwen3-4B-Instruct using 4-bit NF4 quantization. The model will generate concise English or Arabic answers using only the reranked evidence and include validated chunk-ID citations.

In [8]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

GENERATION_MODEL_NAME = (
    "Qwen/Qwen3-4B-Instruct-2507"
)

generation_quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading:", GENERATION_MODEL_NAME)

generation_tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_MODEL_NAME
)

generation_model = (
    AutoModelForCausalLM.from_pretrained(
        GENERATION_MODEL_NAME,
        quantization_config=generation_quantization,
        device_map="auto",
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
)

generation_model.eval()

if generation_tokenizer.pad_token_id is None:
    generation_tokenizer.pad_token_id = (
        generation_tokenizer.eos_token_id
    )

generation_memory_gb = (
    generation_model.get_memory_footprint()
    / (1024 ** 3)
)

print("\nGeneration model loaded successfully.")
print("Model device:", generation_model.device)
print(
    "Loaded in 4-bit:",
    getattr(
        generation_model,
        "is_loaded_in_4bit",
        False,
    ),
)
print(
    "Model memory footprint:",
    f"{generation_memory_gb:.2f} GB",
)

Loading: Qwen/Qwen3-4B-Instruct-2507


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]


Generation model loaded successfully.
Model device: cuda:0
Loaded in 4-bit: True
Model memory footprint: 2.42 GB


## 5. Grounded Qwen Generator Adapter

The generator receives a question and reranked evidence chunks. It must answer in the question’s language, use only the supplied evidence, include chunk-ID citations and refuse unsupported questions.

In [9]:
from src.generation.prompt_builder import (
    build_grounded_messages,
)

# Remove sampling parameters because generation is deterministic.
generation_model.generation_config.temperature = None
generation_model.generation_config.top_p = None
generation_model.generation_config.top_k = None


class QwenGeneratorAdapter:
    def __init__(
        self,
        tokenizer,
        model,
        max_new_tokens: int = 180,
    ) -> None:
        self.tokenizer = tokenizer
        self.model = model
        self.max_new_tokens = max_new_tokens
        self.last_latency_ms = 0.0
        self.last_answer = ""

    def __call__(
        self,
        question: str,
        evidence_chunks: Sequence,
    ) -> str:
        if not question.strip():
            raise ValueError("Question cannot be empty.")

        if not evidence_chunks:
            raise ValueError(
                "Evidence chunks cannot be empty."
            )

        torch.cuda.synchronize()
        start_time = perf_counter()

        messages = build_grounded_messages(
            question=question,
            evidence_chunks=evidence_chunks,
        )

        model_inputs = (
            self.tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
        )

        model_inputs = {
            name: tensor.to(self.model.device)
            for name, tensor in model_inputs.items()
        }

        input_token_count = (
            model_inputs["input_ids"].shape[1]
        )

        with torch.inference_mode():
            generated_tokens = self.model.generate(
                **model_inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        answer_tokens = generated_tokens[
            0,
            input_token_count:,
        ]

        answer = self.tokenizer.decode(
            answer_tokens,
            skip_special_tokens=True,
        ).strip()

        torch.cuda.synchronize()

        self.last_latency_ms = (
            perf_counter() - start_time
        ) * 1000

        self.last_answer = answer
        return answer


qwen_generator = QwenGeneratorAdapter(
    tokenizer=generation_tokenizer,
    model=generation_model,
)

print("Qwen grounded generator adapter created.")

Qwen grounded generator adapter created.


In [10]:
test_question = (
    "How many annual leave days do "
    "full-time employees receive?"
)

test_evidence = [
    chunk
    for chunk in chunks
    if chunk.chunk_id == "HR-EN-001-CH-003"
]

test_answer = qwen_generator(
    test_question,
    test_evidence,
)

print("Question:")
print(test_question)

print("\nEvidence chunk:")
print(test_evidence[0].chunk_id)

print("\nGrounded answer:")
print(test_answer)

print(
    "\nGeneration latency:",
    f"{qwen_generator.last_latency_ms:.2f} ms",
)

Question:
How many annual leave days do full-time employees receive?

Evidence chunk:
HR-EN-001-CH-003

Grounded answer:
Full-time employees receive 24 working days of annual leave after completing one year of service [HR-EN-001-CH-003].

Generation latency: 3216.00 ms


## 6. End-to-End Enterprise RAG Pipeline

Connect multilingual dense retrieval, BGE cross-encoder reranking, grounded Qwen generation and citation validation into one safe Arabic–English RAG pipeline.

In [11]:
rag_pipeline = EnterpriseRAGPipeline(
    retriever=dense_retriever,
    reranker=bge_reranker,
    generator=qwen_generator,
    candidate_k=5,
    evidence_k=3,
)

print("End-to-end RAG pipeline created.")

End-to-end RAG pipeline created.


In [12]:
english_question = (
    "How many annual leave days do "
    "full-time employees receive?"
)

torch.cuda.synchronize()
pipeline_start_time = perf_counter()

english_response = rag_pipeline.answer(
    english_question
)

torch.cuda.synchronize()

total_latency_ms = (
    perf_counter() - pipeline_start_time
) * 1000

print("Question:")
print(english_response.question)

print("\nAnswer:")
print(english_response.answer)

print("\nLanguage:")
print(english_response.language)

print("\nRetrieved candidates:")
print(english_response.candidate_chunk_ids)

print("\nReranked evidence:")
print(english_response.evidence_chunk_ids)

print("\nCitations:")
print(english_response.citations)

print("\nUnsupported citations:")
print(english_response.unsupported_citations)

print("\nCitations valid:")
print(english_response.citations_valid)

print("\nRefused:")
print(english_response.refused)

print("\nSafety blocked:")
print(english_response.safety_blocked)

print("\nLatency breakdown:")
print(
    "Dense retrieval:",
    f"{dense_retriever.last_latency_ms:.2f} ms",
)
print(
    "BGE reranking:",
    f"{bge_reranker.last_latency_ms:.2f} ms",
)
print(
    "Qwen generation:",
    f"{qwen_generator.last_latency_ms:.2f} ms",
)
print(
    "Total pipeline:",
    f"{total_latency_ms:.2f} ms",
)

Question:
How many annual leave days do full-time employees receive?

Answer:
Full-time employees receive 24 working days of annual leave after completing one year of service [HR-EN-001-CH-003].

Language:
en

Retrieved candidates:
('HR-EN-001-CH-003', 'HR-EN-001-CH-002', 'HR-EN-001-CH-004', 'HR-AR-001-CH-005', 'HR-AR-001-CH-004')

Reranked evidence:
('HR-EN-001-CH-003', 'HR-AR-001-CH-005', 'HR-AR-001-CH-004')

Citations:
('HR-EN-001-CH-003',)

Unsupported citations:
()

Citations valid:
True

Refused:
False

Safety blocked:
False

Latency breakdown:
Dense retrieval: 18.32 ms
BGE reranking: 29.40 ms
Qwen generation: 3041.21 ms
Total pipeline: 3090.40 ms


### Arabic End-to-End Test

Verify Arabic retrieval, reranking, same-language generation and citation validation.

In [13]:
arabic_question = (
    "ما الحد الأقصى لتكلفة الفندق "
    "لليلة الواحدة؟"
)

torch.cuda.synchronize()
pipeline_start_time = perf_counter()

arabic_response = rag_pipeline.answer(
    arabic_question
)

torch.cuda.synchronize()

arabic_total_latency_ms = (
    perf_counter() - pipeline_start_time
) * 1000

print("السؤال:")
print(arabic_response.question)

print("\nالإجابة:")
print(arabic_response.answer)

print("\nلغة السؤال:")
print(arabic_response.language)

print("\nالمستندات المسترجعة:")
print(arabic_response.candidate_chunk_ids)

print("\nالأدلة بعد إعادة الترتيب:")
print(arabic_response.evidence_chunk_ids)

print("\nالاستشهادات:")
print(arabic_response.citations)

print("\nالاستشهادات غير المدعومة:")
print(arabic_response.unsupported_citations)

print("\nالاستشهادات صحيحة:")
print(arabic_response.citations_valid)

print("\nتم رفض الإجابة:")
print(arabic_response.refused)

print("\nتم حظر الإجابة للحماية:")
print(arabic_response.safety_blocked)

print("\nزمن التنفيذ:")

print(
    "الاسترجاع:",
    f"{dense_retriever.last_latency_ms:.2f} ms",
)

print(
    "إعادة الترتيب:",
    f"{bge_reranker.last_latency_ms:.2f} ms",
)

print(
    "توليد الإجابة:",
    f"{qwen_generator.last_latency_ms:.2f} ms",
)

print(
    "إجمالي المسار:",
    f"{arabic_total_latency_ms:.2f} ms",
)

السؤال:
ما الحد الأقصى لتكلفة الفندق لليلة الواحدة؟

الإجابة:
الحد الأقصى لتكلفة الفندق لليلة الواحدة هو 450 ريالاً سعودياً، ما لم تتم الموافقة مسبقاً على مبلغ أعلى [HR-AR-001-CH-003].

لغة السؤال:
ar

المستندات المسترجعة:
('HR-AR-001-CH-003', 'HR-AR-001-CH-002', 'HR-AR-001-CH-005', 'HR-AR-001-CH-004', 'HR-AR-001-CH-001')

الأدلة بعد إعادة الترتيب:
('HR-AR-001-CH-003', 'HR-AR-001-CH-002', 'HR-AR-001-CH-004')

الاستشهادات:
('HR-AR-001-CH-003',)

الاستشهادات غير المدعومة:
()

الاستشهادات صحيحة:
True

تم رفض الإجابة:
False

تم حظر الإجابة للحماية:
False

زمن التنفيذ:
الاسترجاع: 13.59 ms
إعادة الترتيب: 28.94 ms
توليد الإجابة: 5326.55 ms
إجمالي المسار: 5369.81 ms


### Cross-Language End-to-End Test

Ask an Arabic question whose answer exists only in the English document. The pipeline must retrieve and rerank the English evidence, answer in Arabic and cite the English chunk.

In [14]:
cross_language_question = (
    "كم عدد أيام الإجازة السنوية للموظف؟"
)

torch.cuda.synchronize()
pipeline_start_time = perf_counter()

cross_language_response = rag_pipeline.answer(
    cross_language_question
)

torch.cuda.synchronize()

cross_language_total_ms = (
    perf_counter() - pipeline_start_time
) * 1000

print("السؤال:")
print(cross_language_response.question)

print("\nالإجابة:")
print(cross_language_response.answer)

print("\nلغة الإجابة المطلوبة:")
print(cross_language_response.language)

print("\nترتيب الاسترجاع الأولي:")
print(
    cross_language_response.candidate_chunk_ids
)

print("\nالأدلة بعد إعادة الترتيب:")
print(
    cross_language_response.evidence_chunk_ids
)

print("\nالاستشهادات:")
print(cross_language_response.citations)

print("\nالاستشهادات غير المدعومة:")
print(
    cross_language_response.unsupported_citations
)

print("\nالاستشهادات صحيحة:")
print(
    cross_language_response.citations_valid
)

print("\nتم رفض الإجابة:")
print(cross_language_response.refused)

print("\nتم حظر الإجابة للحماية:")
print(cross_language_response.safety_blocked)

print("\nزمن التنفيذ:")
print(
    "الاسترجاع:",
    f"{dense_retriever.last_latency_ms:.2f} ms",
)
print(
    "إعادة الترتيب:",
    f"{bge_reranker.last_latency_ms:.2f} ms",
)
print(
    "توليد الإجابة:",
    f"{qwen_generator.last_latency_ms:.2f} ms",
)
print(
    "إجمالي المسار:",
    f"{cross_language_total_ms:.2f} ms",
)

السؤال:
كم عدد أيام الإجازة السنوية للموظف؟

الإجابة:
يُمنح الموظف 24 يومًا عملًا من الإجازة السنوية بعد إكمال سنة خدمة [HR-EN-001-CH-003].

لغة الإجابة المطلوبة:
ar

ترتيب الاسترجاع الأولي:
('HR-AR-001-CH-004', 'HR-AR-001-CH-003', 'HR-AR-001-CH-005', 'HR-EN-001-CH-003', 'HR-AR-001-CH-002')

الأدلة بعد إعادة الترتيب:
('HR-EN-001-CH-003', 'HR-AR-001-CH-004', 'HR-AR-001-CH-005')

الاستشهادات:
('HR-EN-001-CH-003',)

الاستشهادات غير المدعومة:
()

الاستشهادات صحيحة:
True

تم رفض الإجابة:
False

تم حظر الإجابة للحماية:
False

زمن التنفيذ:
الاسترجاع: 14.25 ms
إعادة الترتيب: 28.44 ms
توليد الإجابة: 3712.39 ms
إجمالي المسار: 3755.72 ms


### Unsupported-Question Safety Test

Ask a question that is not answered by the available documents. The system should return the predefined refusal response without inventing facts or citations.

In [15]:
unsupported_question = (
    "What is the company's maternity leave policy?"
)

expected_refusal = (
    "I could not find sufficient evidence in the "
    "provided documents to answer this question."
)

torch.cuda.synchronize()
pipeline_start_time = perf_counter()

unsupported_response = rag_pipeline.answer(
    unsupported_question
)

torch.cuda.synchronize()

unsupported_total_ms = (
    perf_counter() - pipeline_start_time
) * 1000

print("Question:")
print(unsupported_response.question)

print("\nExpected refusal:")
print(expected_refusal)

print("\nPipeline response:")
print(unsupported_response.answer)

print("\nExact refusal returned:")
print(
    unsupported_response.answer
    == expected_refusal
)

print("\nRetrieved candidates:")
print(
    unsupported_response.candidate_chunk_ids
)

print("\nReranked evidence:")
print(
    unsupported_response.evidence_chunk_ids
)

print("\nCitations:")
print(unsupported_response.citations)

print("\nUnsupported citations:")
print(
    unsupported_response.unsupported_citations
)

print("\nCitations valid:")
print(unsupported_response.citations_valid)

print("\nRefused:")
print(unsupported_response.refused)

print("\nSafety blocked:")
print(unsupported_response.safety_blocked)

print("\nLatency breakdown:")
print(
    "Dense retrieval:",
    f"{dense_retriever.last_latency_ms:.2f} ms",
)
print(
    "BGE reranking:",
    f"{bge_reranker.last_latency_ms:.2f} ms",
)
print(
    "Qwen generation:",
    f"{qwen_generator.last_latency_ms:.2f} ms",
)
print(
    "Total pipeline:",
    f"{unsupported_total_ms:.2f} ms",
)

Question:
What is the company's maternity leave policy?

Expected refusal:
I could not find sufficient evidence in the provided documents to answer this question.

Pipeline response:
I could not find sufficient evidence in the provided documents to answer this question.

Exact refusal returned:
True

Retrieved candidates:
('HR-EN-001-CH-001', 'HR-EN-001-CH-003', 'HR-AR-001-CH-001', 'HR-EN-001-CH-002', 'HR-EN-001-CH-005')

Reranked evidence:
('HR-EN-001-CH-003', 'HR-AR-001-CH-001', 'HR-EN-001-CH-001')

Citations:
()

Unsupported citations:
()

Citations valid:
True

Refused:
True

Safety blocked:
False

Latency breakdown:
Dense retrieval: 20.11 ms
BGE reranking: 33.46 ms
Qwen generation: 2437.36 ms
Total pipeline: 2491.97 ms


## 7. End-to-End Evaluation

Evaluate English, Arabic, cross-language retrieval and unsupported-question refusal. Record answer quality, citation integrity, safety behavior and component latency.

In [16]:
import pandas as pd

from src.generation.prompt_builder import (
    REFUSAL_RESPONSES,
    detect_question_language,
)

evaluation_cases = [
    {
        "test": "English grounded answer",
        "question": (
            "How many annual leave days do "
            "full-time employees receive?"
        ),
        "expected_language": "en",
        "expected_phrase": "24",
        "expected_citation": "HR-EN-001-CH-003",
        "expected_refusal": False,
    },
    {
        "test": "Arabic grounded answer",
        "question": (
            "ما الحد الأقصى لتكلفة الفندق "
            "لليلة الواحدة؟"
        ),
        "expected_language": "ar",
        "expected_phrase": "450",
        "expected_citation": "HR-AR-001-CH-003",
        "expected_refusal": False,
    },
    {
        "test": "Arabic question → English evidence",
        "question": (
            "كم عدد أيام الإجازة السنوية للموظف؟"
        ),
        "expected_language": "ar",
        "expected_phrase": "24",
        "expected_citation": "HR-EN-001-CH-003",
        "expected_refusal": False,
    },
    {
        "test": "English unsupported refusal",
        "question": (
            "What is the company's maternity "
            "leave policy?"
        ),
        "expected_language": "en",
        "expected_phrase": None,
        "expected_citation": None,
        "expected_refusal": True,
    },
    {
        "test": "Arabic unsupported refusal",
        "question": (
            "ما هي سياسة إجازة الأمومة في الشركة؟"
        ),
        "expected_language": "ar",
        "expected_phrase": None,
        "expected_citation": None,
        "expected_refusal": True,
    },
]

quality_rows = []
latency_rows = []

for case in evaluation_cases:
    torch.cuda.synchronize()
    start_time = perf_counter()

    response = rag_pipeline.answer(
        case["question"]
    )

    torch.cuda.synchronize()

    total_latency = (
        perf_counter() - start_time
    ) * 1000

    answer_language = detect_question_language(
        response.answer
    )

    language_passed = (
        answer_language
        == case["expected_language"]
    )

    if case["expected_refusal"]:
        content_passed = (
            response.answer
            == REFUSAL_RESPONSES[
                case["expected_language"]
            ]
            and response.refused
        )

        citation_policy_passed = (
            not response.citations
            and response.citations_valid
        )

        safe_behavior_passed = response.refused

    else:
        content_passed = (
            case["expected_phrase"]
            in response.answer
            and not response.refused
        )

        citation_policy_passed = (
            case["expected_citation"]
            in response.citations
            and response.citations_valid
            and not response.unsupported_citations
        )

        safe_behavior_passed = (
            not response.safety_blocked
        )

    overall_passed = all(
        [
            language_passed,
            content_passed,
            citation_policy_passed,
            safe_behavior_passed,
        ]
    )

    quality_rows.append(
        {
            "Test": case["test"],
            "Question Language": (
                case["expected_language"]
            ),
            "Answer Language": answer_language,
            "Top Evidence": (
                response.evidence_chunk_ids[0]
                if response.evidence_chunk_ids
                else "None"
            ),
            "Citations": (
                ", ".join(response.citations)
                if response.citations
                else "None"
            ),
            "Refused": response.refused,
            "Safety Blocked": (
                response.safety_blocked
            ),
            "Language Passed": language_passed,
            "Content/Refusal Passed": (
                content_passed
            ),
            "Citation Policy Passed": (
                citation_policy_passed
            ),
            "Overall Passed": overall_passed,
        }
    )

    latency_rows.append(
        {
            "Test": case["test"],
            "Retrieval (ms)": round(
                dense_retriever.last_latency_ms,
                2,
            ),
            "Reranking (ms)": round(
                bge_reranker.last_latency_ms,
                2,
            ),
            "Generation (ms)": round(
                qwen_generator.last_latency_ms,
                2,
            ),
            "Total (ms)": round(
                total_latency,
                2,
            ),
        }
    )

quality_table = pd.DataFrame(quality_rows)
latency_table = pd.DataFrame(latency_rows)

display(quality_table)
display(latency_table)

passed_tests = int(
    quality_table["Overall Passed"].sum()
)

total_tests = len(quality_table)

print(
    "\nEnd-to-end checks passed:",
    f"{passed_tests}/{total_tests}",
)

,Test,Question Language,Answer Language,Top Evidence,Citations,Refused,Safety Blocked,Language Passed,Content/Refusal Passed,Citation Policy Passed,Overall Passed
0,English grounded answer,en,en,HR-EN-001-CH-003,HR-EN-001-CH-003,False,False,True,True,True,True
1,Arabic grounded answer,ar,ar,HR-AR-001-CH-003,HR-AR-001-CH-003,False,False,True,True,True,True
2,Arabic question → English evidence,ar,ar,HR-EN-001-CH-003,HR-EN-001-CH-003,False,False,True,True,True,True
3,English unsupported refusal,en,en,HR-EN-001-CH-003,None,True,False,True,True,True,True
4,Arabic unsupported refusal,ar,ar,HR-AR-001-CH-004,None,True,False,True,True,True,True


,Test,Retrieval (ms),Reranking (ms),Generation (ms),Total (ms)
0,English grounded answer,125.92,148.47,5783.00,6057.63
1,Arabic grounded answer,11.06,15.73,4638.75,4667.14
2,Arabic question → English evidence,10.79,14.94,4366.58,4393.57
3,English unsupported refusal,10.66,16.10,1671.10,1698.01
4,Arabic unsupported refusal,11.25,21.60,2400.36,2433.38



End-to-end checks passed: 5/5


## Results

The end-to-end Arabic–English Enterprise RAG pipeline passed all five functional checks:

| Capability | Result |
|---|---|
| English grounded question answering | Passed |
| Arabic grounded question answering | Passed |
| Arabic question with English evidence | Passed |
| English unsupported-question refusal | Passed |
| Arabic unsupported-question refusal | Passed |
| Evidence-only citation validation | Passed |

The cross-language test demonstrated the value of reranking. For the Arabic annual-leave question, the correct English evidence initially appeared at dense-retrieval rank 4. The BGE cross-encoder moved it to rank 1, after which Qwen generated a correct Arabic answer citing the English chunk.

Median component latency across the five tests was approximately:

- Dense retrieval: 11.06 ms
- BGE reranking: 16.10 ms
- Qwen generation: 4,366.58 ms
- End-to-end pipeline: 4,393.57 ms

Generation was the primary latency bottleneck.

## Safety Behaviour

The pipeline:

- Generates answers in the question's language.
- Uses only reranked evidence.
- Requires factual answers to include chunk-ID citations.
- Rejects citations that are not present in the supplied evidence.
- Returns localized refusals for unsupported questions.
- Prevents the reranker from introducing chunks that were not retrieved.

## Limitations

This is a focused portfolio demonstration using two synthetic enterprise documents, ten chunks and five evaluation questions. The `5/5` result represents functional test performance and should not be interpreted as production accuracy.

Current limitations include:

- Small synthetic evaluation dataset.
- Text documents only.
- In-memory embeddings rather than a persistent vector database.
- Citation validation checks chunk IDs but does not perform sentence-level entailment verification.
- No authentication, document-level access control or production monitoring.
- GPU-dependent local model inference.

## Next Steps

1. Build a larger bilingual evaluation dataset.
2. Add PDF and DOCX ingestion.
3. Introduce a persistent vector database.
4. Add sentence-level faithfulness evaluation.
5. Expose the pipeline through FastAPI.
6. Build a bilingual Streamlit interface.
7. Add Docker deployment and GitHub Actions.